# 04 - Modeling & Evaluation (D1-D9)
**Study this notebook most carefully.** It explains encoding, scaling, leakage protection, model comparison and evaluation.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
SEED = 42
import time, joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from lightgbm import LGBMRegressor
df=pd.read_csv(ROOT/'data/processed/master_train.csv')

## 1. X and y - remove leakage columns
`yield_tons_per_ha` is the answer, so it becomes **y**. `price_birr_per_quintal` is deliberately excluded because the instructions forbid price as a yield-model feature.

In [11]:
TARGET='yield_tons_per_ha'
DROP=['plot_id',TARGET,'price_birr_per_quintal','weather_months_found']
features=[c for c in df.columns if c not in DROP]
X=df[features]; y=df[TARGET]
print('Features:',features)
assert 'price_birr_per_quintal' not in X
assert any(c.startswith('season_') for c in X)

Features: ['region', 'crop_type', 'survey_year', 'planting_month', 'altitude_m', 'rainfall_mm_season', 'farm_size_ha', 'fertilizer_kg_per_ha', 'improved_seed_used', 'pest_disease_flag', 'soil_quality_index', 'labor_days_per_ha', 'distance_to_market_km', 'season_avg_temp_c', 'season_rainfall_mm', 'season_extreme_heat_days', 'season_temp_deviation_c', 'planting_month_num', 'fertilizer_seed_interaction', 'rainfall_gap_mm']


## 2. Validation split BEFORE learned preprocessing
This is crucial for leakage prevention.

In [12]:
X_train,X_val,y_train,y_val=train_test_split(X,y,test_size=0.20,random_state=SEED)
cat=[c for c in features if not pd.api.types.is_numeric_dtype(X[c])]
num=[c for c in features if c not in cat]
print('Train:',X_train.shape,'Validation:',X_val.shape)

Train: (12072, 20) Validation: (3018, 20)


## 3. Encoding + standardization inside a Pipeline
- Numeric missing values: median learned from training only.
- Categorical: most-frequent imputation + OneHotEncoder.
- StandardScaler is used for Ridge; tree models do not need scaling.
Because these steps are inside the pipeline, cross-validation learns them separately inside each training fold.

In [13]:
def make_preprocessor(scale=False):
    numeric=[('imputer',SimpleImputer(strategy='median'))]
    if scale: numeric.append(('scaler',StandardScaler()))
    return ColumnTransformer([
        ('num',Pipeline(numeric),num),
        ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)
    ])

## D1-D2. Baselines + at least 3 stronger model families

In [14]:
models={
 'Mean':Pipeline([('pre',make_preprocessor()),('model',DummyRegressor())]),
 'Ridge':Pipeline([('pre',make_preprocessor(True)),('model',Ridge())]),
 'RandomForest':Pipeline([('pre',make_preprocessor()),('model',RandomForestRegressor(n_estimators=250,min_samples_leaf=2,n_jobs=-1,random_state=SEED))]),
 'HistGradientBoosting':Pipeline([('pre',make_preprocessor()),('model',HistGradientBoostingRegressor(max_iter=300,learning_rate=.06,random_state=SEED))]),
 'LightGBM':Pipeline([('pre',make_preprocessor()),('model',LGBMRegressor(n_estimators=500,learning_rate=.04,random_state=SEED,verbosity=-1))])
}
# Executed final results are saved here so you can study them instantly:
pd.read_csv(ROOT/'reports/model_comparison.csv').round(4)

,model,RMSE,MAE,R2,train_seconds
0,LightGBM,0.4600,0.3354,0.8940,0.8695
1,Hist Gradient Boosting,0.4748,0.3440,0.8871,0.7808
2,Random Forest,0.5225,0.3832,0.8633,5.3808
3,Ridge,0.8990,0.6714,0.5952,0.0402
4,Mean baseline,1.4131,1.1042,-0.0001,0.0388


## D3. 5-fold cross-validation
The final project used 5 folds for the tuned winner and runner-up. Mean +/- standard deviation tells us both accuracy and stability.

In [15]:
print((ROOT/'reports/D_model_evaluation.md').read_text().split('## D4')[0].split('## D3')[1])

 Cross-validation
Final tuned LightGBM 5-fold RMSE: **0.4690 +/- 0.0114**. Runner-up Hist Gradient Boosting: **0.4737 +/- 0.0116**.




## D4. Out-of-time test
Train on 2021-2023, validate on 2024. This is harder and more realistic than a random split when future seasons may differ.

## D5. Weather ablation
Train the same model twice: without engineered weather and with weather. This directly answers whether the weather join improves prediction.

## D6. Hyperparameter tuning
We use `RandomizedSearchCV` rather than guessing parameters. Tuning happens only on training/CV data - never on the leaderboard test set.

In [16]:
print((ROOT/'reports/D_model_evaluation.md').read_text())

# D - Modeling & Evaluation

## D1 Baselines

| model         |   RMSE |    MAE |      R2 |   train_seconds |
|:--------------|-------:|-------:|--------:|----------------:|
| Ridge         | 0.899  | 0.6714 |  0.5952 |          0.0402 |
| Mean baseline | 1.4131 | 1.1042 | -0.0001 |          0.0388 |

## D2 Model comparison

| model                  |   RMSE |    MAE |      R2 |   train_seconds |
|:-----------------------|-------:|-------:|--------:|----------------:|
| LightGBM               | 0.46   | 0.3354 |  0.894  |          0.8695 |
| Hist Gradient Boosting | 0.4748 | 0.344  |  0.8871 |          0.7808 |
| Random Forest          | 0.5225 | 0.3832 |  0.8633 |          5.3808 |
| Ridge                  | 0.899  | 0.6714 |  0.5952 |          0.0402 |
| Mean baseline          | 1.4131 | 1.1042 | -0.0001 |          0.0388 |

Winner before tuning: **LightGBM**.

## D3 Cross-validation
Final tuned LightGBM 5-fold RMSE: **0.4690 +/- 0.0114**. Runner-up Hist Gradient Boosting: **0.4737 +

## D7-D8. Error analysis and response

In [17]:
display(pd.read_csv(ROOT/'reports/error_by_crop.csv'))
display(pd.read_csv(ROOT/'reports/error_by_region.csv'))
display(pd.read_csv(ROOT/'reports/top10_errors.csv'))

,crop_type,mean,median,count
0,barley,0.287960,0.204661,620
1,maize,0.461215,0.350351,593
2,sorghum,0.320819,0.249647,590
3,teff,0.260323,0.186020,604
4,wheat,0.350577,0.266662,611


,region,mean,median,count
0,Amhara,0.379456,0.313246,594
1,Oromia,0.383275,0.303200,589
2,SNNPR,0.340095,0.252044,569
3,Somali,0.190083,0.132693,632
4,Tigray,0.391110,0.305385,634


,crop_type,region,altitude_m,season_avg_temp_c,actual,predicted,abs_error,residual
0,sorghum,Tigray,2523.475903,16.666667,0.428278,3.251020,2.822742,-2.822742
1,barley,Oromia,1542.252201,19.700000,0.190279,2.694929,2.504649,-2.504649
2,maize,Amhara,1214.213523,17.250000,3.870792,6.353097,2.482305,-2.482305
3,wheat,Tigray,2713.405469,15.100000,6.970028,4.719546,2.250482,2.250482
4,sorghum,Tigray,1205.874983,16.275000,4.570541,2.458585,2.111956,2.111956
5,maize,Tigray,1606.918087,21.350000,8.283826,6.371464,1.912362,1.912362
6,maize,Tigray,1575.960768,21.350000,7.413014,5.510747,1.902266,1.902266
7,maize,Oromia,1541.497940,19.200000,7.913190,6.015238,1.897953,1.897953
8,wheat,Amhara,2471.000023,18.375000,3.021381,4.865029,1.843647,-1.843647
9,wheat,Amhara,2634.792035,15.300000,8.211242,6.375808,1.835435,1.835435


## D9. Plain-language metric
The report converts MAE/RMSE into tons/ha, percentage of average yield, and approximate birr/ha so a cooperative manager can understand it.

## Final model and submission
After model selection is completely finished, the chosen pipeline is refit on **all training rows** and used once on the 3,750 test rows. The final model is `models/final_model.joblib`; predictions are `submission/team_crop_champions_submission.csv`.

## D8. Response to findings
D7 showed several large over/under-predictions. We test a train-only percentile clipping response and keep it only if held-out RMSE improves.

## D9. Plain-language metric
Translate final RMSE/MAE to tons/ha, percent of mean yield, and approximate birr/ha using average cleaned price.


In [ ]:
# ============================================================
# D8 — RESPONSE TO ERROR ANALYSIS
# Did clipping extreme predictions improve the final model?
# ============================================================

from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import joblib

# Load the selected final pipeline and reproduce validation predictions.
# We clone and fit only on X_train/y_train so validation data stays unseen.
final_pipe = joblib.load(ROOT / "models/final_model.joblib")
check_model = clone(final_pipe)
check_model.fit(X_train, y_train)

val_pred = check_model.predict(X_val)

# Original validation error
original_rmse = mean_squared_error(y_val, val_pred) ** 0.5

# Clipping limits learned ONLY from the training target.
lower_limit = y_train.quantile(0.005)
upper_limit = y_train.quantile(0.995)

clipped_pred = np.clip(val_pred, lower_limit, upper_limit)
clipped_rmse = mean_squared_error(y_val, clipped_pred) ** 0.5
rmse_change = clipped_rmse - original_rmse

print("\n" + "=" * 72)
print("D8 — RESPONSE TO ERROR ANALYSIS")
print("=" * 72)

print("\nQUESTION")
print("Can we reduce the largest prediction errors by clipping extreme predictions?")

print("\nMETHOD")
print("• Fit the selected LightGBM pipeline on the training split only.")
print("• Predict the held-out validation split.")
print("• Clip predictions using the 0.5th and 99.5th percentiles of y_train only.")
print(f"• Training-only clipping range: {lower_limit:.3f} to {upper_limit:.3f} tons/ha.")

print("\nRESULT")
print(f"Original validation RMSE : {original_rmse:.4f} tons/ha")
print(f"Clipped validation RMSE  : {clipped_rmse:.4f} tons/ha")
print(f"Change in RMSE           : {rmse_change:+.4f} tons/ha")

if clipped_rmse < original_rmse:
    improvement = (original_rmse - clipped_rmse) / original_rmse * 100
    print(f"\nDECISION: KEEP CLIPPING")
    print(f"Clipping improved RMSE by {improvement:.2f}%.")
else:
    worsening = (clipped_rmse - original_rmse) / original_rmse * 100
    print(f"\nDECISION: REJECT CLIPPING")
    print(f"Clipping made RMSE {worsening:.2f}% worse.")
    print("The original LightGBM predictions are therefore retained.")

print("\nCONCLUSION")
print(
    "The error-analysis response was tested rather than assumed. "
    "Because clipping did not improve held-out validation accuracy, "
    "the final model remains unchanged."
)


# ============================================================
# D9 — PLAIN-LANGUAGE METRIC
# Translate prediction error into farm and business meaning
# ============================================================

mae = mean_absolute_error(y_val, val_pred)
mean_yield = y.mean()
avg_price = df["price_birr_per_quintal"].mean()

rmse_pct = 100 * original_rmse / mean_yield
mae_pct = 100 * mae / mean_yield

# 1 ton = 10 quintals
approx_mae_value = mae * 10 * avg_price

print("\n" + "=" * 72)
print("D9 — PLAIN-LANGUAGE INTERPRETATION")
print("=" * 72)

print("\nYIELD ERROR")
print(f"Average yield in training data : {mean_yield:.2f} tons/ha")
print(f"RMSE                           : {original_rmse:.3f} tons/ha")
print(f"RMSE as % of mean yield        : {rmse_pct:.1f}%")
print(f"MAE                            : {mae:.3f} tons/ha")
print(f"MAE as % of mean yield         : {mae_pct:.1f}%")

print("\nAPPROXIMATE ECONOMIC MEANING")
print(f"Average cleaned crop price     : {avg_price:,.0f} birr/quintal")
print(f"Approx. MAE value per hectare  : {approx_mae_value:,.0f} birr/ha")

print("\nFOR A FARM COOPERATIVE MANAGER")
print(
    f"On average, the model misses actual yield by about {mae:.2f} tons per "
    f"hectare, which is around {mae_pct:.1f}% of the average yield. "
    f"The RMSE is {original_rmse:.2f} tons per hectare ({rmse_pct:.1f}% of "
    f"mean yield), showing the typical size of larger prediction errors. "
    f"Using the average cleaned market price of about {avg_price:,.0f} birr "
    f"per quintal, the MAE corresponds to an approximate value difference "
    f"of {approx_mae_value:,.0f} birr per hectare. This should be treated as "
    f"an indication of prediction uncertainty, not a guaranteed financial loss."
)
